# DAR Data Augmentation Pipeline

This notebook implements the Data Augmentation for Recipe retrieval (DAR) framework.
It uses:
1. **Ollama (Llama 3)** to synthesize 30-word visual imaginations of recipes.
2. **Segment Anything Model (SAM)** to crop food items out of the background.

We use the official Meta `segment_anything` package.

In [1]:
!uv add segment_anything opencv-python requests tqdm pillow matplotlib

import os
from pathlib import Path

# Download SAM weights if they don't exist
weights_dir = Path("sam_weights")
weights_dir.mkdir(exist_ok=True)
sam_checkpoint = weights_dir / "sam_vit_h_4b8939.pth"
if not sam_checkpoint.exists():
    print("Downloading SAM ViT-H checkpoint...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O {sam_checkpoint}
    print("Download complete.")

Resolved 108 packages in 1ms
Checked 102 packages in 1ms


In [2]:
import json
import torch
import cv2
import numpy as np
import requests
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from segment_anything import sam_model_registry, SamPredictor

# --- CONFIGURATION ---
DEBUG = True  # Set to False to process the entire dataset (Overnight run)
DEBUG_LIMIT = 5

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'paper_experiment' else Path.cwd()
DATASET_PATH = PROJECT_ROOT / 'data/datasets/paired_dataset_70k.json'
if not DATASET_PATH.exists():
    DATASET_PATH = PROJECT_ROOT / 'data/datasets/paired_dataset3.json' # Fallback to alternate 74k name

OUTPUT_JSON_PATH = Path("paired_dataset_dar.json")
AUGMENTED_IMAGES_DIR = PROJECT_ROOT / 'data/images/sam_cropped'
AUGMENTED_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

/home/s22imc10262/data/NLP/hackathon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## 1. Load Dataset & Models

In [3]:
# Load SAM
print("Loading Segment Anything Model...")
sam = sam_model_registry["vit_h"](checkpoint=str(sam_checkpoint))
sam.to(device=device)
predictor = SamPredictor(sam)

# Load Recipe Dataset
print(f"Loading dataset from {DATASET_PATH}...")
with open(DATASET_PATH, 'r') as f:
    dataset = json.load(f)

if DEBUG:
    print(f"[DEBUG MODE] Taking only the first {DEBUG_LIMIT} items.")
    dataset = dataset[:DEBUG_LIMIT]
else:
    print(f"Loaded {len(dataset)} pairs for full processing.")

Loading Segment Anything Model...
Loading dataset from /data/s22imc10262/NLP/hackathon/data/datasets/paired_dataset_70k.json...
[DEBUG MODE] Taking only the first 5 items.


## 2. Augmentation Functions

In [4]:
def augment_text_with_ollama(title, ingredients, instructions):
    """
    Uses Ollama to generate a ~30-word visual description of the final dish.
    """
    prompt = (
        f"Recipe Name: {title}\n"
        f"Ingredients: {ingredients}\n"
        f"Instructions: {instructions[:500]}...\n\n"
        "Based on the recipe above, write a brief, highly visual description (around 30 words) "
        "of what the final prepared dish looks like on a plate. "
        "Focus only on its visual appearance, colors, textures, and plating. "
        "Do not include the recipe instructions or greeting.\n"
        "Description:"
    )
    
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False
    }
    
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=30)
        response.raise_for_status()
        return response.json().get("response", "").strip()
    except Exception as e:
        print(f"Ollama request failed: {e}")
        return f"{title} with {str(ingredients)[:100]}" # Fallback

def extract_food_with_sam(image_path, output_path):
    """
    Uses SAM to segment the central food object and crop the background.
    Returns True if successful.
    """
    if not Path(image_path).is_absolute():
        clean_path = str(image_path).replace('../../', '')
        image_path = str(PROJECT_ROOT / clean_path)
        
    if not Path(image_path).exists():
        return False

    # Read image
    image = cv2.imread(image_path)
    if image is None:
        return False
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Predict using the center point of the image as the positive prompt
    predictor.set_image(image_rgb)
    h, w = image.shape[:2]
    input_point = np.array([[w // 2, h // 2]])
    input_label = np.array([1]) # Positive point
    
    masks, scores, _ = predictor.predict(
        point_coords=input_point,
        point_labels=input_label,
        multimask_output=True,
    )
    
    # Pick the mask with the highest score
    best_mask = masks[np.argmax(scores)]
    
    # Create an image with a white background where the mask is False
    result_img = image_rgb.copy()
    result_img[~best_mask] = [255, 255, 255] # White out background
    
    # Save the augmented image
    result_bgr = cv2.cvtColor(result_img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(output_path), result_bgr)
    return True

## 3. Run Pipeline

In [ ]:
augmented_dataset = []

print("Starting augmentation pipeline...")
for i, item in enumerate(tqdm(dataset, desc="Augmenting pairs")):
    orig_img_path = item['image_path']
    recipe_id = str(item.get('recipe_id', ''))
    title = item.get('recipe_title') or item.get('recipe_name') or ''
    ingredients = item.get('ingredients', '')
    instructions = item.get('directions', '') # Try getting instructions if available in item
    
    # Define output path for SAM image
    img_filename = Path(orig_img_path).name
    aug_img_path = AUGMENTED_IMAGES_DIR / f"sam_{img_filename}"
    
    # 1. Image Augmentation
    success = extract_food_with_sam(orig_img_path, aug_img_path)
    if not success:
        print(f"Warning: SAM failed or image missing for {orig_img_path}. Skipping.")
        continue
        
    # 2. Text Augmentation
    aug_text = augment_text_with_ollama(title, ingredients, instructions)
    
    # Compile
    augmented_dataset.append({
        "recipe_id": recipe_id,
        "image_path": orig_img_path,
        "aug_image_path": str(aug_img_path.absolute()),
        "recipe_title": title,
        "ingredients": ingredients,
        "aug_visual_text": aug_text
    })
    
    if DEBUG and i == 0:
        print(f"\nSample Visual Description:\n{aug_text}\n")

# Save new dataset
with open(OUTPUT_JSON_PATH, 'w') as f:
    json.dump(augmented_dataset, f, indent=2)

print(f"\nPipeline complete. Augmented dataset saved to {OUTPUT_JSON_PATH}")

Starting augmentation pipeline...


Augmenting pairs:  20%|██        | 1/5 [00:04<00:17,  4.38s/it]


Sample Visual Description:
A golden-brown chicken breast "abalone" shell glistens on the plate, its tender flesh flaked to resemble the prized mollusk's texture. A drizzle of rich, amber-hued clam juice sauce glimmers alongside, contrasting with the creamy white plate and garnished with a sprig of fresh parsley for a pop of freshness.



Augmenting pairs: 100%|██████████| 5/5 [00:08<00:00,  1.70s/it]


Pipeline complete. Augmented dataset saved to paired_dataset_dar.json
